# Chemical space visualization

In this exercise, you will build your own chemical space visualizations using datasets from the Zenodo record at https://zenodo.org/records/18678356. The record contains a data/ folder containing *outputs of different molecular generators*. This data is the output used in our recent paper [cite:fil], which you are welcome to skim through for more details. Look at the data for the **Glucocorticoid receptor** (focus only on the *_10k* sets for the dissimilarity split, use all output sets indexed 0-4) and compare the chemical spaces covered by the *DrugEx_GT_epsilon_0.6* and *GB_GA_mut_r_0.5*. The goal is to <u>compare how these two different generators explore chemical space relative to the reference chemistry</u>. The *reference chemistry set* can be found in the ChEMBL database for the Glucocorticoid receptor (CHEMBL2034).

### Learning goals

- Load molecular datasets into Python.
- Convert molecules into **descriptor vectors** or **fingerprints** with *RDKit*.
- Use *scikit-learn* to compute **PCA**, **t-SNE** or other visualizations.
- Compare reference chemistry and generated chemistry.
- Interpret differences in coverage, clustering, overlap, and outliers.

### Recommended dataset selection

1. Download the archive from the Zenodo page and inspect the data/ folder to find the designated data sets above.

2. You can use the *ChEMBL Python API* to download the reference chemistry for the Glucocorticoid receptor.

### Suggested workflow

1. Create a Python environment with RDKit, scikit-learn, pandas, matplotlib, and seaborn.

2. Inspect the downloaded files and determine whether each selected dataset is stored as SMILES tables, CSV files, or SDF files.

3. Load the molecules with RDKit. For SMILES-based files, read the table and convert the SMILES strings with Chem.MolFromSmiles. For SDF files, use Chem.SDMolSupplier.

4. Remove invalid molecules and duplicates so that all datasets are processed consistently.

5. Generate a common molecular representation for all selected datasets.

> i. Start with **Morgan fingerprints** such as radius 2 and 2048 bits because they are widely used in chemical similarity analysis.

> ii. Optionally build a second representation based on **physicochemical descriptors** such as molecular weight, logP, TPSA, hydrogen bond donor count, hydrogen bond acceptor count, and rotatable bond count (can be calculated with RDKit).

8. For non-binary descriptor vectors, *standardize* the features before dimensionality reduction.

9. Use scikit-learn **PCA** to generate a first 2D plot.

10. Use scikit-learn **t-SNE** to generate a second 2D plot from the same representation.

11. Color each point by *dataset origin* so that the sets can be compared directly.

12. Save the plots and write a short (two page max) report of what is similar and what is different across the datasets (reports will be discussed during the practical in person workshop).

> However, treat the above as a suggested workflow. You are welcome to explore other representations, similarity measures, and projection methods. You are also welcome to use coding agents to help you with the implementation, but make sure to understand what the code is doing and be able to explain it in your report. You can also consider making the visualizations interactive with Plotly or Bokeh to display molecules on hover, but this is optional.

## 0. Setup and datasets

- Create a Python environment with RDKit, scikit-learn, pandas, matplotlib, and seaborn.

In [ ]:
# Download RDKit and the ChEMBL API
!pip -q install RDKit
!pip -q install chembl_webresource_client

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator, Descriptors, rdMolDescriptors, QED
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Draw import MolsToGridImage

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

- Download the necessary datasets:
  - From the Zenodo record (or Discord): outputs of different molecular generators for the Glucocorticoid receptor (steroid receptor for glucocorticoids such as cortisol). The outputs are generated from **DrugEx GT** (DGM-based, deep generative model) and from **GB_GA** (GA-based, genetic algorithm-based).
  - Using the <u>ChEMBL Python API</u>: reference chemistry (known active molecules) for the Glucocorticoid receptor

In [ ]:
# Mount Google Drive to get the molecular generator datasets
from google.colab import drive
drive.mount('/content/drive')

### a. Molecular Generators Datasets

*Note* on **Dataset1 (molecular generators)**: I'll concatenate all outputs from each of the two generators, saving the index of the original file for each set.

In [ ]:
import glob

# DATASET1: Upload the datasets from the molecular generators as Pandas dataframes
# List of datasets in each folder
files_drugex = glob.glob('drive/MyDrive/data/DrugEx_GT_epsilon_0.6_10k/*')
files_gbga = glob.glob('drive/MyDrive/data/GB_GA_mut_r_0.5_10k/*')

# For each molecular generator, upload data as a dataframe
def upload_data(files_list):
    print("Uploading", files_list)
    df_list = [] # List of datasets in files_list
    for file in files_list:
      # Create a DataFrame
      df = pd.DataFrame(columns=['smiles', 'dataset_id'])
      # Read the file
      df['smiles'] = pd.read_csv(file)
      # Add a column to record the index of the source file
      parts = file.split('_')
      df['dataset_id'] = int(parts[-3])
      # Append to the list of dataframes for data in files_list
      df_list.append(df)
    # Concatenate the data
    df_files = pd.concat(df_list, ignore_index=True)
    return df_files

df_drugex = upload_data(files_drugex)
print(len(df_drugex), "entries for DrugEx datasets")
df_gbga = upload_data(files_gbga)
print(len(df_gbga), "entries for GB_GA datasets")

### b. Reference Chemistry Dataset

### ChEMBL

The ChEMBL database is organized into different categories (called resources), such as
- `target`: Information about biological receptors, enzymes, and proteins
- `molecule`: Information about chemical structures and properties
- `activity`: Experimental measurements showing how a molecule interacts with a target (e.g., $IC_{50}$, $K_i$, inhibition percentages)

Some attributes associated to a bioactivity record (`activity`):
- `molecule_chembl_id`: unique identifier for the compound
- `canonical_smiles`: canonical SMILES
- `standard_type`, `standard_relation`, `standard_value`, `standard_units`: name of the measurement or assay type (e.g. IC50), mathematical operator relating the value to the measurement (>, <, =), numerical magnitude, unit of measure
- `pchembl_value`: pre-calculated, negative log scale value ($-\log_{10}$ molar) provided by ChEMBL for absolute concentrations (like IC50 or Ki where units are molar). For example, a 1 micromolar ($1 \mu M$) concentration becomes a pChEMBL value of 6
- `assay_chembl_id`: identifier for the laboratory assay or paper experiment
- `data_validity_comment`: notes added by the ChEMBL curators if a data point has potential issue, None if everything looks clean


*Note* on **Dataset2 (reference chemistry)**: I retrieve only IC50, Ki, EC50 and AC50 assay (leaving out kinetic data and less specific activity metrics) and keep only data with standard_relationship '=' to exclude ambiguous entries.

In [ ]:
# DATASET2: Upload the reference chemistry for the glucocorticoid receptor from ChEMBL
from chembl_webresource_client.new_client import new_client

# ChEMBL ID of the glucocorticoid receptor
target_chembl_id = 'CHEMBL2034'
# Establish a communication channel with the activity resource in ChEMBL
activity = new_client.activity
# Fields to retrieve
desired_columns = [
    'molecule_chembl_id', 'canonical_smiles',
    'assay_type', 'standard_type', 'standard_value', 'standard_units',
    'assay_description', 'data_validity_comment'
]
# Retrieve active compounds for the given receptor, together with selected attributes
activities = activity.filter(
    target_chembl_id=target_chembl_id,
    standard_type__in=["IC50", "Ki", "EC50", "AC50"],
    standard_relation__exact = '='
).only(desired_columns)

In [ ]:
# Convert to a Pandas DataFrame
df_refchem = pd.DataFrame(list(activities))
print(f"Retrieved {len(df_refchem)} total raw records for this receptor.")

df_refchem.head()

In [ ]:
# Clean it up (delete redundant columns)
df_refchem = df_refchem[[col for col in desired_columns]]

# Save it to a file
#df_refchem.to_csv('drive/MyDrive/data/reference_chemistry.csv', index=False)

### c. Approved Drugs / Clinical Candidates

Upload the **approved drugs / clinical candidates** for the glucocorticoid receptor.

In [ ]:
# ChEMBL ID of the glucocorticoid receptor
target_chembl_id = 'CHEMBL2034'

# Establish a communication channel with the mechanism and molecule resources in ChEMBL
mech = new_client.mechanism
molecule = new_client.molecule

# Retrieve drug data for the glucocorticoid receptor
mechs = mech.filter(
    target_chembl_id=target_chembl_id
).only([
    'molecule_chembl_id', 'mechanism_of_action', 'action_type',
    'mechanism_comment', 'disease_efficacy'
])
# Convert to DataFrame
df_mech = pd.DataFrame(list(mechs))
print(f"Retrieved {len(df_mech)} total mechanisms for this receptor.")
df_mech.head()

In [ ]:
# Extract molecular structures
mol_chembl_ids = df_mech["molecule_chembl_id"].unique().tolist()

# Fetch structural and phase data including SMILES
mol_query = molecule.filter(molecule_chembl_id__in=mol_chembl_ids).only(
    "molecule_chembl_id",
    "pref_name",
    "max_phase",
    "molecule_structures"
)

# Convert to DataFrame
df_mol = pd.DataFrame(list(mol_query))
print(f"Retrieved {len(df_mol)} total molecules for this receptor.")
df_mol.head()

In [ ]:
# Merge the mechanism and molecule data
df_drug = pd.merge(df_mol, df_mech, on="molecule_chembl_id")

# Clean it up (keep only the SMILES in the column molecule_structures)
df_drug['canonical_smiles'] = [d.get("canonical_smiles") if isinstance(d, dict) else None for d in df_drug['molecule_structures']]
df_drug.drop(columns=["molecule_structures"], inplace=True)

print(f"Retrieved {len(df_drug)} total drugs for this receptor.")
df_drug.head()

## 1. Data cleaning & exploration

- **Validate SMILES strings**: for SMILE-based files, convert the SMILES strings with `Chem.MolFromSmiles`; if a structure is invalid, RDKit fails to parse it

- Inspect and eventually exclude entries with a warning comment from curators, for the ChEMBL reference chemistry

- **Canonicalization**: generate canonical SMILES with `Chem.MolToSmiles`

- **Remove duplicates**: same molecule (check canonical SMILES); salts, mixtures (check numeber of fragments, then keep the largest if appropriate and check again canonical SMILES)

- **Neutralize**: the goal is a structural comparison to molecules generated *de novo* as neutral, hence as long as all non-neutral molecules are salts I'll neutralize them.

### SMILES validation

In [ ]:
# List of datasets to clean
data = [(df_drugex, 'df_drugex'), (df_gbga, 'df_gbga'), (df_refchem, 'df_refchem'), (df_drug, 'df_drug')]

# Add a column to flag entries to exclude
for dataset in data:
  df = dataset[0]
  df['flag_exclude'] = False # Do I exclude the entry?
  df['exclude_comment'] = None # Why do I exclude the entry? None if flag_exclude False

print("df_drugex columns:", list(df_drugex.columns))
print("df_gbga columns:", list(df_gbga.columns))
print("df_refchem columns:", list(df_refchem.columns))
print("df_drug columns:", list(df_drug.columns))

In [ ]:
# 1. Validate SMILES
def smiles_to_mol(smiles):
    if pd.isna(smiles):
        return None
    try:
        return Chem.MolFromSmiles(str(smiles))
    except Exception:
        return None

for dataset in data:
  df = dataset[0]
  df_name = dataset[1]
  # Convert from SMILES to Mol object
  smiles_col = "smiles" if "smiles" in df.columns else "canonical_smiles"
  df["mol"] = df[smiles_col].apply(smiles_to_mol)
  df["valid_smiles"] = df["mol"].notna()
  # Check if there are any invalid SMILES
  print(df_name, "has", (~df["valid_smiles"]).sum(), "invalid SMILES")

*Comment*: Only the data from ChEMBL have invalid entries. The following cells inspect these entries.

In [ ]:
df_refchem[df_refchem['valid_smiles'] == False]

In [ ]:
df_drug[df_drug['valid_smiles'] == False]

The entries with invalid SMILES didn't have a SMILES from ChEMBL, so I exclude them.

In [ ]:
df_refchem.loc[df_refchem['valid_smiles'] == False, "flag_exclude"] = True
df_refchem.loc[df_refchem['valid_smiles'] == False, "exclude_comment"] = "Invalid SMILES"
df_refchem[df_refchem["valid_smiles"] == False]

df_drug.loc[df_drug['valid_smiles'] == False, "flag_exclude"] = True
df_drug.loc[df_drug['valid_smiles'] == False, "exclude_comment"] = "Invalid SMILES"
df_drug[df_drug["valid_smiles"] == False]


### data_validity_comment

In [ ]:
# For the reference chemistry, inspect the entries with a data validity comment from curators
print(f"{len(df_refchem[df_refchem["data_validity_comment"].notna()])} entries have a data_validity_comment")

# Exclude those entries
df_refchem.loc[df_refchem["data_validity_comment"].notna(), "flag_exclude"] = True
df_refchem.loc[df_refchem["data_validity_comment"].notna(), "exclude_comment"] = "Curators flagged as outside typical range"

df_refchem[df_refchem["data_validity_comment"].notna()]

Curators warn about the activity values being "Outside typical range". I inspect the other entries to check what is the typical range for EC50 and IC50, which are the assay types flagged by this warning (and for the other assays for completeness).

In [ ]:
assays = ["IC50", "Ki", "EC50", "AC50"]

def assay_range(df_refchem, assay):
  # Filter for the assay_type "assay"
  df_filt = df_refchem[df_refchem['standard_type'] == assay]
  # Print min and max
  units = df_filt['standard_units'].unique()
  for u in units:
    df_filt_u = df_filt[df_filt['standard_units'] == u]
    print(f"{assay}: {df_filt_u['standard_value'].min()} {u} - {df_filt_u['standard_value'].max()} {u}")

for i,assay in enumerate(assays):
  assay_range(df_refchem[df_refchem["flag_exclude"] == False], assay)

EC50 value expressed in percentage points towards an error in the data point, so I exclude it.

In [ ]:
df_refchem.loc[df_refchem["standard_units"] == "%", "flag_exclude"] = True
df_refchem.loc[df_refchem["standard_units"] == "%", "exclude_comment"] = "Data error or mislabeling"
df_refchem[df_refchem["standard_units"] == "%"]

In [ ]:
# 2. Generate canonical smiles
def canonical_smiles(mol):
    if mol is None:
        return np.nan
    return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)

for dataset in data:
  df = dataset[0]
  df_name = dataset[1]
  df["canonical_smiles_rdkit"] = df["mol"].apply(canonical_smiles)
  # Compare the canonical SMILES to the original SMILES to see if there are any differences
  smiles_col = "smiles" if "smiles" in df.columns else "canonical_smiles"
  df["smiles_equal"] = df["canonical_smiles_rdkit"] == df[smiles_col]

*Notes:*
- `df_drugex` has 88 entries where the SMILES generated from RDKit differs from the given one
- `df_gbga`has 2470 entries where the SMILES generated by RDKit differs from the given one - one difference is the representation of aromaticity
- `df_refchem` has 1 entry where the SMILES generated by RDKit differs from the given one
- `df_drug` has no entries where the SMILES generated by RDKit differs from the given one

In [ ]:
# Recap on what has been excluded for each dataset
for dataset in data:
  df = dataset[0]
  df_name = dataset[1]
  print(f"{df_name} has {len(df)} entries and {df['flag_exclude'].sum()} entries to exclude")

For `df_refchem` I exclude 5 invalid SMILES, 12 data curator warnings, and 1 wrong data point. For `df_drug` I exclude 1 invalid SMILES.

### Remove duplicates
- Duplicate molecules

In [ ]:
# 3. Remove duplicates: identical molecules, then salts / mixtures
# Check number of duplicates, using RDKit Mol objects, canonical SMILES from ChEMBL and canonical SMILES from RDKit for comparison
for dataset in data:
  df = dataset[0]
  df_name = dataset[1]
  nuniq = df["mol"].nunique()
  nnan = df["mol"].isna().sum()
  print(f"{df_name} has {len(df)} entries and {nuniq} unique RDKit Mol - {len(df)-nuniq-nnan} duplicates")
print("Mol objects cannot be compared directly!\n---")

for dataset in data:
  df = dataset[0]
  df_name = dataset[1]
  nuniq = df["canonical_smiles_rdkit"].nunique()
  nnan = df["canonical_smiles_rdkit"].isna().sum()
  print(f"{df_name} has {len(df)} entries and {nuniq} unique canonical RDKit SMILES - {len(df)-nuniq-nnan} duplicates")

print("---")
for dataset in data[2:]:
  df = dataset[0]
  df_name = dataset[1]
  nuniq = df['canonical_smiles'].nunique()
  nnan = df["canonical_smiles"].isna().sum()
  print(f"{df_name} has {len(df)} entries and {nuniq} unique canonical ChEMBL SMILES - {len(df)-nuniq-nnan} duplicates")

print("---")
for dataset in data[2:]:
  df = dataset[0]
  df_name = dataset[1]
  nuniq = df["molecule_chembl_id"].nunique()
  nnan = df["molecule_chembl_id"].isna().sum()
  print(f"{df_name} has {len(df)} entries and {nuniq} unique ChEMBL IDs - {len(df)-nuniq-nnan} duplicates")

- ChEMBL IDs may vary across database versions: different IDs can point to the same structure, and the same ID can point to different structures across different versions - but that should not be a problem in this case as all data were downloaded in the same session
- <u>`Mol` objects cannot be compared directly</u>: the comparison falls back to comparing memory addresses
- To deduplicate, I use the SMILES generated by RDKit from the `Mol` objects (so that I am sure that the algorithm is the same across all entries) - Using InChiKey to deduplicate would probably be a better solution

<hr>

- ChEMBL and RDKit canonical SMILES agree on the number of duplicates
- Canonical SMILES and ChEMBL IDs do NOT agree on the number of duplicates: two more duplicates according to ChEMBL ID: these two have invalid SMILES, hence RDKit SMILES based deduplication ignored them.


In [ ]:
# Check duplicates according to ChEMBL ID but not canonical SMILES
df_refchem[~df_refchem["valid_smiles"]]
# The two more duplicates according to ChEMBL IDs have invalid SMILES

In [ ]:
# Flag molecules that are identical according to RDKit canonical SMILES (keep the first occurrence)
for dataset in data:
  df = dataset[0]
  df_name = dataset[1]
  df["is_duplicate_canonical"] = df.duplicated(subset=["canonical_smiles_rdkit"], keep="first") & df["canonical_smiles_rdkit"].notna()
  print(f"{df_name} has {df["is_duplicate_canonical"].sum()} RDKit SMILES duplicates")
  # Exclude occurrences that are not the first (I don't need activity data so I'll just keep the first occurrence - If I did, I would have to look at the values to check if they are coherent before grouping them)
  df.loc[df["is_duplicate_canonical"], "flag_exclude"] = True
  df.loc[df["is_duplicate_canonical"] & df["exclude_comment"].isna(), "exclude_comment"] = "Duplicate canonical SMILES - before removing salt / mixtures and neutralizing"

In [ ]:
# Recap on what has been excluded for each dataset
for dataset in data:
  df = dataset[0]
  df_name = dataset[1]
  print(f"{df_name} has {len(df)} entries and {df['flag_exclude'].sum()} entries to exclude")

- Check number of fragments (for salts and mixtures)

In [ ]:
# Count fragments to identify salt and mixtures
def count_fragments(mol):
    if mol is None:
        return np.nan
    return len(Chem.GetMolFrags(mol))

for dataset in data:
  df = dataset[0]
  df_name = dataset[1]
  df["n_fragments"] = df["mol"].apply(count_fragments)
  df["has_multiple_fragments"] = df["n_fragments"] > 1
  print(f"{df_name} has {df["has_multiple_fragments"].sum()} entries with more than one fragment")

In [ ]:
# Inspect the entries with more than one fragment
print("df_refchem")
print(df_refchem[df_refchem["has_multiple_fragments"]][["molecule_chembl_id", "n_fragments"]])

# Inspect the SMILES
#print(df_refchem.loc[1247, "canonical_smiles_rdkit"])

multfrag_refchem = df_refchem[df_refchem["has_multiple_fragments"]].copy()
MolsToGridImage(
    multfrag_refchem["mol"].tolist(),
    legends=multfrag_refchem["molecule_chembl_id"].tolist(),
    molsPerRow=4,
    subImgSize=(250, 200)
)

In [ ]:
print("\ndf_drug")
print(df_drug[df_drug["has_multiple_fragments"]][["molecule_chembl_id", "n_fragments"]])

# Inspect the SMILES
#print(df_drug.loc[63, "canonical_smiles_rdkit"])

multfrag_drug = df_drug[df_drug["has_multiple_fragments"]].copy()
MolsToGridImage(
    multfrag_drug["mol"].tolist(),
    legends=multfrag_drug["molecule_chembl_id"].tolist(),
    molsPerRow=4,
    subImgSize=(250, 200)
)

All the structures with more than one fragment are salts of a main structure, hence it is appropriate to keep only the largest fragment.

In [ ]:
# Keep the largest fragment
fragment_chooser = rdMolStandardize.LargestFragmentChooser()

def largest_fragment_mol(mol):
    if mol is None:
        return None
    try:
        return fragment_chooser.choose(mol) # choose the largest fragment
    except Exception:
        return mol

for dataset in data:
  df = dataset[0]
  df_name = dataset[1]
  df["parent_mol"] = df["mol"].apply(largest_fragment_mol)
  df["parent_smiles"] = df["parent_mol"].apply(canonical_smiles)

In [ ]:
# Inspect the entries with more than one fragment

# Inspect the SMILES
#print(df_refchem.loc[1247, "canonical_smiles_rdkit"])

multfrag_refchem = df_refchem[df_refchem["has_multiple_fragments"]].copy()
MolsToGridImage(
    multfrag_refchem["parent_mol"].tolist(),
    legends=multfrag_refchem["molecule_chembl_id"].tolist(),
    molsPerRow=4,
    subImgSize=(250, 200)
)

In [ ]:
# Inspect the SMILES
#print(df_drug.loc[63, "canonical_smiles_rdkit"])

multfrag_drug = df_drug[df_drug["has_multiple_fragments"]].copy()
MolsToGridImage(
    multfrag_drug["parent_mol"].tolist(),
    legends=multfrag_drug["molecule_chembl_id"].tolist(),
    molsPerRow=4,
    subImgSize=(250, 200)
)

### Check neutrality

In [ ]:
# Check the charge of all entries
def formal_charge(mol):
    if mol is None:
        return None
    return Chem.GetFormalCharge(mol)

for dataset in data:
  df = dataset[0]
  df_name = dataset[1]
  df["charge"] = df["parent_mol"].apply(formal_charge)
  print(f"{df_name} has {len(df[(df['charge'] != 0) & df['charge'].notna()])} charged entries.")

In [ ]:
# Inspect selected charged entries
print("df_drug")
print(df_drug[(df_drug["charge"] != 0) & df_drug['charge'].notna()][["molecule_chembl_id", "charge"]])

charged_drug = df_drug[(df_drug["charge"] != 0) & df_drug['charge'].notna()].copy()
MolsToGridImage(
    charged_drug["parent_mol"].tolist(),
    legends=charged_drug["molecule_chembl_id"].tolist(),
    molsPerRow=4,
    subImgSize=(250, 200)
)

In [ ]:
# Inspect selected charged entries
print("df_drugex")
print(df_drugex[(df_drugex["charge"] != 0) & df_drugex['charge'].notna()][["charge"]].head(8))

charged_drugex = df_drugex[(df_drugex["charge"] != 0) & df_drugex['charge'].notna()].head(8).copy() # only first 8 entries
MolsToGridImage(
    charged_drugex["parent_mol"].tolist(),
    molsPerRow=4,
    subImgSize=(250, 200)
)

In [ ]:
# Neutralize
uncharger = rdMolStandardize.Uncharger()

def neutralize_mol(mol):
    if mol is None:
        return None
    try:
        return uncharger.uncharge(mol)
    except Exception:
        return mol

for dataset in data:
  df = dataset[0]
  df_name = dataset[1]
  df["parent_mol_neut"] = df["parent_mol"].apply(neutralize_mol)
  df["parent_smiles_neut"] = df["parent_mol_neut"].apply(canonical_smiles)
  df["charge_neut"] = df["parent_mol_neut"].apply(formal_charge)

Remaining charges after neutralizing should be only structural charges (e.g. quaternary amine), not protonation states.

In [ ]:
for dataset in data:
  df = dataset[0]
  df_name = dataset[1]
  print(f"{df_name} has {len(df[(df['charge_neut'] != 0) & df['charge_neut'].notna()])} remaining charged entries after neutralization.")

In [ ]:
# Check for duplicates after removing salts / mixtures and neutralizing
# Flag molecules that are identical according to RDKit canonical SMILES (keep the first occurrence)
for dataset in data:
  df = dataset[0]
  df_name = dataset[1]
  df["is_duplicate_parent_neut"] = df.duplicated(subset=["parent_smiles_neut"], keep="first") & df["parent_smiles_neut"].notna()
  print(f"{df_name} has {df["is_duplicate_parent_neut"].sum()} RDKit SMILES duplicates")
  # Exclude occurrences that are not the first (I don't need activity data so I'll just keep the first occurrence - If I did, I would have to look at the values to check if they are coherent before grouping them)
  df.loc[df["is_duplicate_parent_neut"], "flag_exclude"] = True
  df.loc[df["is_duplicate_parent_neut"] & df["exclude_comment"].isna(), "exclude_comment"] = "Duplicate canonical SMILES - after removing salt / mixtures and neutralizing"

In [ ]:
# Recap on what has been excluded for each dataset
for dataset in data:
  df = dataset[0]
  df_name = dataset[1]
  print(f"{df_name} has {len(df)} entries and {df['flag_exclude'].sum()} entries to exclude")

### Clean datasets

In [ ]:
df_drugex_clean = (df_drugex[~df_drugex["flag_exclude"]])[['parent_mol_neut',	'parent_smiles_neut']].copy()
df_gbga_clean = (df_gbga[~df_gbga["flag_exclude"]])[['parent_mol_neut',	'parent_smiles_neut']].copy()
df_refchem_clean = (df_refchem[~df_refchem["flag_exclude"]])[['parent_mol_neut',	'parent_smiles_neut']].copy()
df_drug_clean = (df_drug[~df_drug["flag_exclude"]])[['parent_mol_neut',	'parent_smiles_neut']].copy()

data_clean = [(df_drugex_clean, 'df_drugex'), (df_gbga_clean, 'df_gbga'), (df_refchem_clean, 'df_refchem'), (df_drug_clean, 'df_drug')]

## 2. Descriptors

Calculate **fingerprints** and **physicochemical descriptors** for all datasets.

- **Fingerprints**: Morgan, radius 2, 2048 bit
- **Physicochemical descriptors**:
  - MW, LogP (or cLogP), TPSA, HBD, HBA
  - Rotatable bonds, aromatic ring count, fraction Csp3
  - Number of rings, heteroatom count
  - QED (drug-likeness score)
  - Formal charge, molar refractivity

In [ ]:
# Fingerprints
def morgan_fingerprints(mol, radius=2, nBits=2048):
    if mol is None:
        return None
    mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=nBits)
    fp = mfpgen.GetFingerprintAsNumPy(mol)  # as numpy array for sklearn
    return fp

for dataset in data_clean:
  df = dataset[0]
  df_name = dataset[1]
  df["fp"] = df["parent_mol_neut"].apply(morgan_fingerprints)

In [ ]:
# Descriptors
def calc_descriptors(mol):
    if mol is None:
        return None
    return pd.Series({
        'MW': Descriptors.MolWt(mol),
        'LogP': Descriptors.MolLogP(mol),
        'TPSA': Descriptors.TPSA(mol),
        'HBD': Descriptors.NumHDonors(mol),
        'HBA': Descriptors.NumHAcceptors(mol),
        'RotBonds': Descriptors.NumRotatableBonds(mol),
        'AromaticRings': rdMolDescriptors.CalcNumAromaticRings(mol),
        'FractionCSP3': rdMolDescriptors.CalcFractionCSP3(mol),
        'RingCount': rdMolDescriptors.CalcNumRings(mol),
        'HeteroatomCount': rdMolDescriptors.CalcNumHeteroatoms(mol),
        'QED': QED.qed(mol), # closer to 1 means more drug-like
        'FormalCharge': Chem.GetFormalCharge(mol),
        'MolarRefractivity': Descriptors.MolMR(mol), # higher: larger / more polarizable
    })

for i,dataset in enumerate(data_clean):
  df = dataset[0]
  df_name = dataset[1]
  print(f"Calculating {df_name}")
  descriptor_df = df["parent_mol_neut"].apply(calc_descriptors)
  df = pd.concat([df.reset_index(drop=True), descriptor_df.reset_index(drop=True)], axis=1)
  df = df.round(2)
  data_clean[i] = (df, df_name)  # write back into the list
  print(f"Done {df_name}")

In [ ]:
for i,dataset in enumerate(data_clean):
  df = dataset[0]
  df_name = dataset[1]
  # Save to a file
  #df.drop(columns=["parent_mol_neut"], axis=1).to_csv(f"drive/MyDrive/data/{df_name}_descr.csv", index=False)

"""
# Load files with precalculated descriptors
for i,dataset in enumerate(data_clean):
  df_name = dataset[1]
  df = pd.read_csv(f"drive/MyDrive/data/{df_name}_descr")
  data_clean[i] = (df, df_name)
"""

df_drugex_clean = data_clean[0][0]
df_gbga_clean = data_clean[1][0]
df_refchem_clean = data_clean[2][0]
df_drug_clean = data_clean[3][0]

# Concatenate all datasets for PCA and t-SNE, after adding a flag for origin dataset
for dataset in data_clean:
  df = dataset[0]
  df_name = dataset[1]
  df["source"] = df_name
df_all = pd.concat([df_drugex_clean, df_gbga_clean, df_refchem_clean, df_drug_clean])

#### MW filtering

In [ ]:
# Inspect MW distribution
plt.figure(figsize=(6, 4))
sns.histplot(data=df_all, x='MW', hue='source', bins=50, alpha=0.5, element='step', stat='density', common_norm=False)
plt.axvline(700, color='red', linestyle='--', label='proposed cutoff')
plt.xlabel('MW')
plt.show()

print(df_all['MW'].describe())
print(f"Entries above 700: {(df_all['MW'] > 700).sum()}")
print(f"Entries above 1000: {(df_all['MW'] > 1000).sum()}")

In [ ]:
# Plot in separate histograms
g = sns.FacetGrid(df_all, col='source', height=4, col_wrap=4, sharex=True, sharey=False)
g.map_dataframe(sns.histplot, x='MW', bins=30)
g.set_titles(col_template='{col_name}')

for ax in g.axes.flat:
    ax.axvline(700, color='red', linestyle='--')

plt.tight_layout()
plt.show()

In [ ]:
# Filter out MW > 700 for ref_chem
# Filter out entries in refchem with MW > 1000
df_filtered = df_all[
    (df_all['source'] != 'df_refchem') |
    ((df_all['source'] == 'df_refchem') & (df_all['MW'] < 700))
]

# Save to a file
df_filtered.to_csv('drive/MyDrive/data/df_filtered.csv', index=False)

#### Subset of the dataset

In [ ]:
# Select a subset of the data for computational feasibility
refchem_full = df_filtered[df_filtered['source'] == 'df_refchem']  # keep all
drug_full = df_filtered[df_filtered['source'] == 'df_drug']  # keep all

drugex_sample = df_filtered[df_filtered['source'] == 'df_drugex'].sample(n=3000, random_state=42) # select some
gbga_sample = df_filtered[df_filtered['source'] == 'df_gbga'].sample(n=3000, random_state=42) # select some

df_sample = pd.concat([refchem_full, drug_full, drugex_sample, gbga_sample], ignore_index=True)

print(df_sample['source'].value_counts())  # confirm number of samples for each dataset
print(f"Total samples: {len(df_sample)}")

In [ ]:
# Check range for each descriptor
descriptor_cols = ['MW', 'LogP', 'TPSA', 'HBD', 'RotBonds', 'AromaticRings', 'FractionCSP3']
df_sample[descriptor_cols].describe()

## 3. Chemical Space Visualization: PCA, t-SNE

- **PCA**:
  - *Linear*: finds axes (linear combinations of your original features) that capture maximum variance
  - *Global structure*: preserves overall variance/spread — distances between far-apart points are meaningful
  - *Deterministic*: same input always gives the same output
  - *Interpretable*: you can look at loadings (e.g. "PC1 is mostly MW + LogP") and explain what each axis means chemically
  - *Fast*, scales well

- **t-SNE**:
  - *Non-linear*: models local neighborhood similarity, tries to preserve which points are close together
  - *Local structure*: great at revealing clusters/groups; global distances between clusters are not meaningful
  - *Stochastic*: different runs (different random seeds) can give different-looking layouts, though relative clustering tends to be stable
  - *Not interpretable*: axes have no chemical meaning
  - *Sensitive to hyperparameters*, especially perplexity

### From fingerprints

#### a. PCA

In [ ]:
# PCA
# X values: DataFrame of the features in columns turned to a numpy array
X = np.stack(df_sample['fp'].values)  # shape: (n_samples, 2048)

pca = PCA(n_components=2)
pca_result = pca.fit_transform(X)
# pca_result: numpy array of shape (n_samples, n_components), transformed coordinates of each sample in the PC space

df_sample['PC1_fp'] = pca_result[:, 0]
df_sample['PC2_fp'] = pca_result[:, 1]

print(f"Explained variance: PC1={pca.explained_variance_ratio_[0]:.2%}, PC2={pca.explained_variance_ratio_[1]:.2%}")

In [ ]:
# Plot data in PC space
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df_sample, x='PC1_fp', y='PC2_fp', hue='source', alpha=0.6, s=20)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.title('PCA: reference vs de novo chemistry (GR)')
plt.legend()
plt.tight_layout()

In [ ]:
# Plot each dataset in a separate panel
g = sns.FacetGrid(df_sample, col='source', height=5, col_wrap=4)
g.map_dataframe(sns.scatterplot, x='PC1_fp', y='PC2_fp', alpha=0.6, s=20)
g.set_axis_labels(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',
                   f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
g.set_titles(col_template='{col_name}')
plt.tight_layout()

#### b. t-SNE

In [ ]:
# t-SNE
# Calculate on a subset of the data for computational feasibility
X_fp = np.stack(df_sample['fp'].values)  # shape (n_samples, 2048)

print("Computing t-SNE")

tsne = TSNE(
    n_components=2,
    metric='jaccard',
    init='random',        # required - 'pca' init isn't supported with non-euclidean metrics
    perplexity=30,
    method='barnes_hut',  # approximation required for anything beyond a few thousand points; exact method doesn't scale
    random_state=42,
    n_jobs=1              # use all CPU cores to help with runtime
)
tsne_result = tsne.fit_transform(X_fp)
print("Done computing t-SNE")

df_sample['tSNE1_fp'] = tsne_result[:, 0]
df_sample['tSNE2_fp'] = tsne_result[:, 1]

In [ ]:
# Plot data
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df_sample, x='tSNE1_fp', y='tSNE2_fp', hue='source', alpha=0.6, s=20)
plt.xlabel(f'tSNE1')
plt.ylabel(f'tSNE2')
plt.title('tSNE: reference vs de novo chemistry (GR)')
plt.legend()
plt.tight_layout()

In [ ]:
# Plot each dataset in a separate panel
g = sns.FacetGrid(df_sample, col='source', height=5, col_wrap=4)
g.map_dataframe(sns.scatterplot, x='tSNE1_fp', y='tSNE2_fp', alpha=0.6, s=20)
g.set_axis_labels(f'tSNE1',
                   f'tSNE2')
g.set_titles(col_template='{col_name}')
plt.tight_layout()

In [ ]:
# Plot all df_drug with molecules visualized on hover to check fast what these outliers are
from bokeh.plotting import figure, show
from bokeh.models import HoverTool, ColumnDataSource
from bokeh.io import output_notebook
from rdkit.Chem import Draw
import base64
from io import BytesIO

df_drug_sample = df_sample[df_sample['source'] == 'df_drug'].copy()

output_notebook()

def mol_to_base64(mol, size=(200, 200)):
    if mol is None:
        return None
    img = Draw.MolToImage(mol, size=size)
    buffered = BytesIO()
    img.save(buffered, format="PNG")
    img_str = base64.b64encode(buffered.getvalue()).decode()
    return f"data:image/png;base64,{img_str}"

df_drug_sample['mol_img'] = df_drug_sample['parent_mol_neut'].apply(mol_to_base64)

source = ColumnDataSource(df_drug_sample[['tSNE1_fp', 'tSNE2_fp', 'source', 'mol_img']])

p = figure(width=700, height=500, title="t-SNE: GR approved drugs")
p.scatter('tSNE1_fp', 'tSNE2_fp', source=source, size=8, alpha=0.6)

hover = HoverTool(tooltips="""
    <div>
        <img src="@mol_img" width="150"><br>
        <span>@source</span>
    </div>
""")
p.add_tools(hover)

show(p)

### From Descriptors

In [ ]:
# Check correlation between all descriptors
descriptor_cols_all = ['MW', 'LogP', 'TPSA',
       'HBD', 'HBA', 'RotBonds', 'AromaticRings', 'FractionCSP3', 'RingCount',
       'HeteroatomCount', 'QED', 'FormalCharge', 'MolarRefractivity']

corr = df_sample[descriptor_cols_all].corr() # 1: positive corr, 0: no corr, -1: negative corr
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, vmin=-1, vmax=1,
            annot_kws={'size': 8}, fmt='.2f')

In [ ]:
# Select these descriptors to exclude redundant / highly correlated ones and check correlation between these
descriptor_cols = ['MW', 'LogP', 'TPSA', 'HBD', 'RotBonds', 'AromaticRings', 'FractionCSP3']

corr = df_sample[descriptor_cols].corr() # 1: positive corr, 0: no corr, -1: negative corr
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, vmin=-1, vmax=1,
            annot_kws={'size': 8}, fmt='.2f')

In [ ]:
df_filtered[descriptor_cols].describe()  # check ranges of all selected descriptors

#### a. PCA

In [ ]:
# PCA
# X values: DataFrame of the features in columns turned to a numpy array
X = df_sample[descriptor_cols].values # shape: (n_samples, n_descriptors)
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=2)
pca_result = pca.fit_transform(X_scaled)
# pca_result: numpy array of shape (n_samples, n_components), transformed coordinates of each sample in the PC space

df_sample['PC1_desc'] = pca_result[:, 0]
df_sample['PC2_desc'] = pca_result[:, 1]

print(f"Explained variance: PC1={pca.explained_variance_ratio_[0]:.2%}, PC2={pca.explained_variance_ratio_[1]:.2%}")

In [ ]:
# Plot data in PC space
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df_sample, x='PC1_desc', y='PC2_desc', hue='source', alpha=0.6, s=20)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.title('PCA: reference vs de novo chemistry (GR)')
plt.legend()
plt.tight_layout()

In [ ]:
# Inspect loadings
loadings = pd.DataFrame(
    pca.components_.T,  # transpose: rows = descriptors, columns = PCs
    columns=['PC1', 'PC2'],
    index=descriptor_cols
)
print(loadings)

In [ ]:
# Biplot
fig, ax1 = plt.subplots(figsize=(8, 6))

# scatter on primary axis
sns.scatterplot(data=df_sample, x='PC1_desc', y='PC2_desc', hue='source', alpha=0.4, s=15, ax=ax1)
ax1.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax1.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')

# loadings on a secondary, independently-scaled axis
ax2 = ax1.twinx().twiny()
for i, descriptor in enumerate(descriptor_cols):
    ax2.arrow(0, 0, loadings.iloc[i]['PC1'], loadings.iloc[i]['PC2'],
              color='black', alpha=0.8, head_width=0.03, length_includes_head=True)
    ax2.text(loadings.iloc[i]['PC1']*1.15, loadings.iloc[i]['PC2']*1.15,
              descriptor, fontsize=7, ha='center', color='black')

ax2.set_xlim(-1.2, 1.2)
ax2.set_ylim(-1.2, 1.2)
ax2.set_xticks([])
ax2.set_yticks([])

plt.title('PCA Biplot')
plt.tight_layout()
plt.show()

In [ ]:
# Plot each dataset in a separate panel
g = sns.FacetGrid(df_sample, col='source', height=5, col_wrap=4)
g.map_dataframe(sns.scatterplot, x='PC1_desc', y='PC2_desc', alpha=0.6, s=20)
g.set_axis_labels(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',
                   f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
g.set_titles(col_template='{col_name}')
plt.tight_layout()

In [ ]:
# Plot all df_drug with molecules visualized on hover to check fast what these outliers are
from bokeh.plotting import figure, show
from bokeh.models import HoverTool, ColumnDataSource
from bokeh.io import output_notebook
from rdkit.Chem import Draw
import base64
from io import BytesIO

df_drug_sample = df_sample[df_sample['source'] == 'df_drug'].copy()

output_notebook()

def mol_to_base64(mol, size=(200, 200)):
    if mol is None:
        return None
    img = Draw.MolToImage(mol, size=size)
    buffered = BytesIO()
    img.save(buffered, format="PNG")
    img_str = base64.b64encode(buffered.getvalue()).decode()
    return f"data:image/png;base64,{img_str}"

df_drug_sample['mol_img'] = df_drug_sample['parent_mol_neut'].apply(mol_to_base64)

source = ColumnDataSource(df_drug_sample[['PC1_desc', 'PC2_desc', 'source', 'mol_img']])

p = figure(width=700, height=500, title="PCA: GR approved drugs")
p.scatter('PC1_desc', 'PC2_desc', source=source, size=8, alpha=0.6)

hover = HoverTool(tooltips="""
    <div>
        <img src="@mol_img" width="150"><br>
        <span>@source</span>
    </div>
""")
p.add_tools(hover)

show(p)

In [ ]:
# Plot all de novo data with molecules visualized on hover to check fast what these outliers are
from bokeh.plotting import figure, show
from bokeh.models import HoverTool, ColumnDataSource
from bokeh.io import output_notebook
from rdkit.Chem import Draw
import base64
from io import BytesIO

df_denovo_sample = df_sample[(df_sample['source'] == 'df_drugex') | (df_sample['source'] == 'df_gbga')].copy()

output_notebook()

def mol_to_base64(mol, size=(200, 200)):
    if mol is None:
        return None
    img = Draw.MolToImage(mol, size=size)
    buffered = BytesIO()
    img.save(buffered, format="PNG")
    img_str = base64.b64encode(buffered.getvalue()).decode()
    return f"data:image/png;base64,{img_str}"

df_denovo_sample['mol_img'] = df_denovo_sample['parent_mol_neut'].apply(mol_to_base64)

source = ColumnDataSource(df_denovo_sample[['PC1_desc', 'PC2_desc', 'source', 'mol_img']])

p = figure(width=700, height=500, title="PCA: GR denovo molecules")
p.scatter('PC1_desc', 'PC2_desc', source=source, size=8, alpha=0.6)

hover = HoverTool(tooltips="""
    <div>
        <img src="@mol_img" width="150"><br>
        <span>@source</span>
    </div>
""")
p.add_tools(hover)

show(p)

From the plot above it is clear that there are some outliers in the `df_refchem` dataset. In the following cell these are identified and visually inspected. - **UPDATE**: I filtered out MW > 700 for `df_refchem`

#### b. t-SNE

In [ ]:
# t-SNE
# Calculate on a subset of the data for computational feasibility
X = df_sample[descriptor_cols].values  # use the MW-filtered dataframe
X_scaled = StandardScaler().fit_transform(X)  # scaling

print("Computing t-SNE")
tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_jobs=1)
tsne_result = tsne.fit_transform(X_scaled)
print("Done computing t-SNE")

df_sample['tSNE1_desc'] = tsne_result[:, 0]
df_sample['tSNE2_desc'] = tsne_result[:, 1]

In [ ]:
# Plot data
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df_sample, x='tSNE1_desc', y='tSNE2_desc', hue='source', alpha=0.6, s=20)
plt.xlabel(f'tSNE1')
plt.ylabel(f'tSNE2')
plt.title('tSNE: reference vs de novo chemistry (GR)')
plt.legend()
plt.tight_layout()

In [ ]:
# Plot each dataset in a separate panel
g = sns.FacetGrid(df_sample, col='source', height=5, col_wrap=4)
g.map_dataframe(sns.scatterplot, x='tSNE1_desc', y='tSNE2_desc', alpha=0.6, s=20)
g.set_axis_labels(f'tSNE1',
                   f'tSNE2')
g.set_titles(col_template='{col_name}')
plt.tight_layout()

In [ ]:
# Plot all df_drug with molecules visualized on hover to check fast what these outliers are
from bokeh.plotting import figure, show
from bokeh.models import HoverTool, ColumnDataSource
from bokeh.io import output_notebook
from rdkit.Chem import Draw
import base64
from io import BytesIO

df_drug_sample = df_sample[df_sample['source'] == 'df_drug'].copy()

output_notebook()

def mol_to_base64(mol, size=(200, 200)):
    if mol is None:
        return None
    img = Draw.MolToImage(mol, size=size)
    buffered = BytesIO()
    img.save(buffered, format="PNG")
    img_str = base64.b64encode(buffered.getvalue()).decode()
    return f"data:image/png;base64,{img_str}"

df_drug_sample['mol_img'] = df_drug_sample['parent_mol_neut'].apply(mol_to_base64)

source = ColumnDataSource(df_drug_sample[['tSNE1_desc', 'tSNE2_desc', 'source', 'mol_img']])

p = figure(width=700, height=500, title="t-SNE: GR approved drugs")
p.scatter('tSNE1_desc', 'tSNE2_desc', source=source, size=8, alpha=0.6)

hover = HoverTool(tooltips="""
    <div>
        <img src="@mol_img" width="150"><br>
        <span>@source</span>
    </div>
""")
p.add_tools(hover)

show(p)

In [ ]:
# Save to a file
df_sample.to_csv('drive/MyDrive/data/df_sample.csv', index=False)

## Questions to answer in the report

1. Which datasets *overlap* strongly, and which are clearly separated?

2. Do the generated sets stay close to *reference chemistry*, or does it explore a different region?

3. Which method, *PCA or t-SNE*, makes the cluster structure easier to see?

4. Do you observe *outliers*, and if so, what might explain them?

5. How does the interpretation change when using *descriptors instead of fingerprints*?